Ce projet consite a crée un agent IA(personnal_chef ) qui :
* Prendre les **ingrédients qu’il vous reste dans la cuisine**.
* **Rechercher sur Internet** des recettes utilisant ces ingrédients[Tools].
* **Vous proposer les recettes trouvées**.
* Pouvoir **répondre à vos questions de suivi**, par exemple en gardant en mémoire les échanges précédents, c’est-à-dire qu’il possède une **« mémoire » des exécutions précédentes**.


In [37]:
from dotenv import load_dotenv
load_dotenv()



True

In [38]:
#Recherche internet 
from langchain.tools import tool
from typing import Dict, Any 
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool("web_search")
def web_search(query: str) ->str:
    """Search the web for information."""
    return tavily_client.search(query)


In [39]:
system_prompt = """
    Tu es un chef de cuisine. 
    Tu utilisera les ingrédients se trouvant sur la photo que j'ai upload, pour rechercher sur internet des menues qu'on peut faires avec ses ingrédients.
    Tu vas par la suite me proposer à base de tes recherche des menus.
    Tu gadera en mémoire courte(short-term) les échanges précédents.
    Tu ne repondra qu'au question en lien avec la cuisine car tu es un chef de cuisine."""
    

In [40]:
# Création de l'agent avec le short-term memory 
from langchain.agents import create_agent 
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
model = ChatOpenAI(model="gpt-4.1-mini")
agent = create_agent(
    model = model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver(),
)

In [41]:
#Dit a lagent ce qui se trouve dans notre cuisine 
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?. please answer in french")]},
    config
)

print(response['messages'][-1].content)

Avec tes restes de poulet et de riz, voici quelques idées de menus que tu peux réaliser :

1. Riz sauté au poulet à l'asiatique : fais sauter ton riz avec des morceaux de poulet, des œufs, des petits pois, de la carotte et un peu de sauce soja. C'est simple et savoureux.
2. Poulet au riz façon anti-gaspi : mélange ton poulet et riz avec des champignons, un peu de crème liquide, sel et poivre pour un plat crémeux.
3. Soupe poulet et nouilles : utilise ton poulet pour faire une soupe chaude avec des nouilles, légumes et bouillon.

Veux-tu que je te donne une recette détaillée pour l'une de ces idées ?


In [42]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='I have some leftover chicken and rice. What can I make?. please answer in french', additional_kwargs={}, response_metadata={}, id='8de9fbc6-a90d-4790-ba9d-bc0f1001afe5'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 156, 'total_tokens': 179, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_d00e189ddc', 'id': 'chatcmpl-ERLyhD0hC87v2DmMuMphAF4n94jWO', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0cf85-c5af-72c3-b42e-9f3588937b71-0', tool_calls=[{'name': 'web_search'

In [43]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Hello"
            }
        ]
    },
    config={
        "configurable": {
            "thread_id": "1"
        }
    }
)

In [44]:
print(response["messages"][-1].content)

Bonjour ! Je suis là pour t’aider avec des questions ou idées de cuisine. Veux-tu des suggestions pour un plat avec tes ingrédients, ou as-tu une autre question culinaire ?
